# Create the wave metamodel

Pipeline (settings from ``NorthCarolina.json`` → ``metamodel`` / ``variables``).

1. **Train / load** models (`fit_models`) — MDA + PCA + GP per goal from SnapWave static cases.
2. **Bulk reconstruction** (`reconstruct_bulk`) — canonical `hs` / `tp` / `wd` / `ds`.
3. **Zero-fill** missing forcing (`apply_missing_forcing_zeros`).
4. **Spectral reconstruction** (`reconstruct_spectral`) — wavespectra stats `hs` / `tp` / `tm01` / `dm`.
5. **Save** (`save_spectral`) → `outputs/spectral_metamodel.nc`.

In [1]:
from utils import metamodel as mm
from utils.config import GOALS, METAMODEL, reload

reload()  # pick up edits to NorthCarolina.json without restarting kernel
print(f"goals            : {len(GOALS)}")
print(f"MDA points       : {METAMODEL['num_mda_points']}")
print(f"spectral output  : {mm.spectral_output_path()}")

goals            : 14
MDA points       : 450
spectral output  : /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc


## Parameters

Tweak these while experimenting — especially `time` and `stations` for the
spectral step.

In [2]:
# --- training --------------------------------------------------------------
force_retrain = False  # set True to overwrite existing .pkl models

# --- bulk reconstruction ---------------------------------------------------
time_bulk = "1986-01-20"  # any xarray time selector (month, day, slice, ...)

# --- spectral reconstruction (bottleneck) ------------------------------------
time_spec = None  # None → use full bulk window; or e.g. "1986-01-15"
sites = None  # flexible: int, slice, list, "buoy", "isobath", name
# sites = slice(0, 3)   # quick smoke test on first 3 sites
# sites = -1              # single site by index

## 1 · Train / load models

In [3]:
models = mm.fit_models(force_retrain=force_retrain)
len(models["mda"]), len(models["pca"]), len(models["gp"])

2026-06-17 15:54:29,572 - NorthCarolina.metamodel - INFO - Fitting models for 14 goal(s)  (num_mda_points=450, force_retrain=False)
2026-06-17 15:54:29,780 - NorthCarolina.metamodel - INFO - Goal  1: loaded existing models
2026-06-17 15:54:29,817 - NorthCarolina.metamodel - INFO - Goal  2: loaded existing models
2026-06-17 15:54:29,854 - NorthCarolina.metamodel - INFO - Goal  3: loaded existing models
2026-06-17 15:54:29,887 - NorthCarolina.metamodel - INFO - Goal  4: loaded existing models
2026-06-17 15:54:29,920 - NorthCarolina.metamodel - INFO - Goal  5: loaded existing models
2026-06-17 15:54:29,957 - NorthCarolina.metamodel - INFO - Goal  6: loaded existing models
2026-06-17 15:54:29,989 - NorthCarolina.metamodel - INFO - Goal  7: loaded existing models
2026-06-17 15:54:30,025 - NorthCarolina.metamodel - INFO - Goal  8: loaded existing models
2026-06-17 15:54:30,221 - NorthCarolina.metamodel - INFO - Goal  9: loaded existing models
2026-06-17 15:54:30,256 - NorthCarolina.metamodel

(14, 14, 14)

## 2 · Bulk reconstruction

In [4]:
bulk = mm.reconstruct_bulk(time_bulk, models=models)
print(f"size : {bulk.nbytes * 1e-9:.2f} GB")
bulk

2026-06-17 15:54:30,882 - NorthCarolina.metamodel - INFO - Bulk reconstruction: time='1986-01-20', goals=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], partitions=['bulk', 'sea', 'swell1', 'swell2', 'swell3']
2026-06-17 15:54:30,889 - NorthCarolina.metamodel - INFO - Goal  1 / bulk   : reconstructing 24 / 24 timesteps
2026-06-17 15:54:31,392 - NorthCarolina.metamodel - INFO - Goal  2 / bulk   : reconstructing 24 / 24 timesteps
INFO:ExactGPInterpolation:Preprocessing dataset for prediction
INFO:ExactGPInterpolation:Checking for NaNs in subset data
INFO:ExactGPInterpolation:Preprocessing subset data
INFO:ExactGPInterpolation:Normalizing subset data
INFO:ExactGPInterpolation:Proposed max custom scaler for ds is lower than datapoint
INFO:ExactGPInterpolation:Subset data preprocessed successfully
INFO:ExactGPInterpolation:Predicting target variable: PC1
INFO:ExactGPInterpolation:Predicting target variable: PC2
INFO:ExactGPInterpolation:Predicting target variable: PC3
INFO:ExactGPInterpola

size : 0.01 GB


<xarray.Dataset> Size: 12MB
Dimensions:      (time: 24, goal: 14, partition: 5, sites: 216)
Coordinates:
  * time         (time) datetime64[ns] 192B 1986-01-20 ... 1986-01-20T23:00:00
  * goal         (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    latitude     (goal) float64 112B 36.56 36.12 35.69 ... 33.19 33.0 32.88
    longitude    (goal) float64 112B -75.5 -75.25 -75.06 ... -78.25 -78.62 -79.0
    seapoint     (goal) float64 112B 1.904e+06 1.904e+06 ... 1.903e+06 1.903e+06
  * partition    (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
  * sites        (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
Data variables:
    hs           (goal, partition, time, sites) float64 3MB 0.6175 ... nan
    tp           (goal, partition, time, sites) float64 3MB 7.947 7.947 ... nan
    wd           (goal, partition, time, sites) float64 3MB 140.2 137.0 ... nan
    ds           (goal, partition, time, sites) float64 3MB 24.7 24.7 ... nan

## 2b · Zero-fill missing forcing

Sea/swell partitions can be NaN when a component is absent. The metamodel only
predicts valid timesteps; this step sets those gaps to ``0.0`` before the
spectral sum.

In [5]:
bulk = mm.apply_missing_forcing_zeros(bulk)
bulk

2026-06-17 15:54:36,009 - NorthCarolina.metamodel - INFO - Zero-filled 615168 NaN values in hs, tp, wd, ds


<xarray.Dataset> Size: 12MB
Dimensions:      (time: 24, goal: 14, partition: 5, sites: 216)
Coordinates:
  * time         (time) datetime64[ns] 192B 1986-01-20 ... 1986-01-20T23:00:00
  * goal         (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    latitude     (goal) float64 112B 36.56 36.12 35.69 ... 33.19 33.0 32.88
    longitude    (goal) float64 112B -75.5 -75.25 -75.06 ... -78.25 -78.62 -79.0
    seapoint     (goal) float64 112B 1.904e+06 1.904e+06 ... 1.903e+06 1.903e+06
  * partition    (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
  * sites        (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
Data variables:
    hs           (goal, partition, time, sites) float64 3MB 0.6175 ... 0.0
    tp           (goal, partition, time, sites) float64 3MB 7.947 7.947 ... 0.0
    wd           (goal, partition, time, sites) float64 3MB 140.2 137.0 ... 0.0
    ds           (goal, partition, time, sites) float64 3MB 24.7 24.7 ... 0.0

## 3 · Spectral reconstruction

Start with a **small** `sites` selection above, then scale up.

In [6]:
spectral = mm.reconstruct_spectral(bulk, sites=sites, time=time_spec)
spectral

2026-06-17 15:54:36,053 - NorthCarolina.metamodel - INFO - Spectral reconstruction: 216 site(s), partitions=['sea', 'swell1', 'swell2', 'swell3'], time=None
2026-06-17 15:54:36,053 - NorthCarolina.metamodel - INFO - Spectral site   1 / 216: 5m_10
2026-06-17 15:54:36,284 - NorthCarolina.metamodel - INFO - Spectral site   2 / 216: 5m_11
2026-06-17 15:54:36,464 - NorthCarolina.metamodel - INFO - Spectral site   3 / 216: 5m_12
2026-06-17 15:54:36,644 - NorthCarolina.metamodel - INFO - Spectral site   4 / 216: 5m_13
2026-06-17 15:54:36,826 - NorthCarolina.metamodel - INFO - Spectral site   5 / 216: 5m_14
2026-06-17 15:54:37,006 - NorthCarolina.metamodel - INFO - Spectral site   6 / 216: 5m_15
2026-06-17 15:54:37,184 - NorthCarolina.metamodel - INFO - Spectral site   7 / 216: 5m_16
2026-06-17 15:54:37,363 - NorthCarolina.metamodel - INFO - Spectral site   8 / 216: 5m_17
2026-06-17 15:54:37,542 - NorthCarolina.metamodel - INFO - Spectral site   9 / 216: 5m_18
2026-06-17 15:54:37,722 - NorthCa

<xarray.Dataset> Size: 174kB
Dimensions:      (sites: 216, time: 24)
Coordinates:
  * sites        (sites) object 2kB '5m_10' '5m_11' ... 'buoy_8658120'
    site         (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
  * time         (time) datetime64[ns] 192B 1986-01-20 ... 1986-01-20T23:00:00
Data variables:
    hs           (sites, time) float64 41kB 1.093 1.126 1.164 ... 2.078 2.02
    tp           (sites, time) float32 21kB 8.138 8.242 8.307 ... 7.568 7.529
    tm01         (sites, time) float64 41kB 6.863 6.996 7.089 ... 6.647 6.606
    dm           (sites, time) float64 41kB 133.5 130.3 131.0 ... 197.2 195.5
Attributes:
    globwave_name:          significant_wave_height
    long_name:              Significant height of wind and swell waves
    standard_name:          sea_surface_wave_significant_height
    units:                  m
    valid_max:              32000
    valid_min:              0
    coverage_content_type:  modelResult
    mesh:                   SMC

In [7]:
import numpy as np

spectral_filtered = spectral.where(spectral.hs.max(dim="time") < 10.0, np.nan).dropna(
    dim="sites"
)
spectral_filtered

<xarray.Dataset> Size: 168kB
Dimensions:      (sites: 209, time: 24)
Coordinates:
  * sites        (sites) object 2kB '5m_10' '5m_11' ... 'buoy_8658120'
    site         (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 8kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
  * time         (time) datetime64[ns] 192B 1986-01-20 ... 1986-01-20T23:00:00
Data variables:
    hs           (sites, time) float64 40kB 1.093 1.126 1.164 ... 2.078 2.02
    tp           (sites, time) float32 20kB 8.138 8.242 8.307 ... 7.568 7.529
    tm01         (sites, time) float64 40kB 6.863 6.996 7.089 ... 6.647 6.606
    dm           (sites, time) float64 40kB 133.5 130.3 131.0 ... 197.2 195.5
Attributes:
    globwave_name:          significant_wave_height
    long_name:              Significant height of wind and swell waves
    standard_name:          sea_surface_wave_significant_height
    units:                  m
    valid_max:              32000
    valid_min:              0
    coverage_content_type:  modelResult
    mesh:                   SMC

## 4 · Save

In [8]:
out_path = mm.save_spectral(spectral_filtered)
print(f"wrote {out_path}")
print(f"variables: {list(spectral_filtered.data_vars)}")

2026-06-17 15:55:17,371 - NorthCarolina.metamodel - INFO - Wrote spectral metamodel to /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc (209 site(s), 0.17 MB)


wrote /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc
variables: ['hs', 'tp', 'tm01', 'dm']
